In [ ]:
import jax
import jax.numpy as jnp
from jax import grad, vmap, jit, random
import optax
import matplotlib.pyplot as plt

# 1. High-Performance MLP Architecture
def init_params(layers, key):
    params = []
    keys = random.split(key, len(layers) - 1)
    for i, (in_dim, out_dim) in enumerate(zip(layers[:-1], layers[1:])):
        w_key, b_key = random.split(keys[i])
        params.append({
            'w': random.normal(w_key, (in_dim, out_dim)) * jnp.sqrt(2/in_dim),
            'b': jnp.zeros((out_dim,))
        })
    return params

def forward(params, x):
    # Standard MLP forward pass
    activation = x
    for layer in params[:-1]:
        activation = jnp.tanh(jnp.dot(activation, layer['w']) + layer['b'])
    return jnp.dot(activation, params[-1]['w']) + params[-1]['b']

# 2. JAX-Powered Physics Loss (Biochemical Diffusion)
@jit
def physics_loss(params, x_batch, diffusion_coeff=0.01):
    # x_batch contains [time, position]

    def net_u(t, x):
        # Helper for double differentiation
        return forward(params, jnp.array([t, x]))[0]

    def diffusion_error(t, x):
        # Use jax.grad for automated physics constraints
        u_t = grad(net_u, argnums=0)(t, x)        # First order time derivative
        u_xx = grad(grad(net_u, argnums=1), argnums=1)(t, x) # Second order spatial
        return (u_t - diffusion_coeff * u_xx)**2

    # Vectorize the error calculation across the batch using vmap
    errors = vmap(diffusion_error)(x_batch[:, 0], x_batch[:, 1])
    return jnp.mean(errors)

# 3. Training Loop with Optax and XLA Compilation (jit)
def train(params, iterations=2000):
    optimizer = optax.adam(learning_rate=1e-3)
    opt_state = optimizer.init(params)

    # Collocation points (t, x) for physics enforcement
    key = random.PRNGKey(42)
    x_collocation = random.uniform(key, (1000, 2), minval=0.0, maxval=1.0)

    @jit
    def step(params, opt_state, x_batch):
        # Compute gradient of the loss function
        loss_val, grads = jax.value_and_grad(physics_loss)(params, x_batch)
        updates, opt_state = optimizer.update(grads, opt_state)
        params = optax.apply_updates(params, updates)
        return params, opt_state, loss_val

    print("Starting JAX-compiled training...")
    for i in range(iterations):
        params, opt_state, loss = step(params, opt_state, x_collocation)
        if i % 500 == 0:
            print(f"Iteration {i}, Loss: {loss:.6f}")
    return params

# Execute
layers = [2, 64, 64, 1] # [t, x] -> [Concentration]
params = init_params(layers, random.PRNGKey(0))
trained_params = train(params)

# 4. Visualization of Bio-Diffusion
t_range = jnp.linspace(0, 1, 100)
x_range = jnp.linspace(0, 1, 100)
T, X = jnp.meshgrid(t_range, x_range)
grid = jnp.stack([T.flatten(), X.flatten()], axis=1)
preds = vmap(lambda x: forward(trained_params, x))(grid).reshape(100, 100)

plt.figure(figsize=(8, 6))
plt.contourf(T, X, preds, cmap='viridis')
plt.colorbar(label='Drug Concentration')
plt.title("JAX-PINN: Modeled Biochemical Diffusion")
plt.xlabel("Time (t)")
plt.ylabel("Position (x)")
plt.show()